In [14]:
# Data Handling
import pandas as pd
import numpy as np

# NLP
import nltk
import re
import string

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
# Evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Save Model
import joblib

In [15]:
import pandas as pd
import numpy as np

import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\melgn\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\melgn\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [16]:
fake = pd.read_csv("fake.csv")
true = pd.read_csv("true.csv")

print(fake.head())
print(true.head())

print(fake.shape)
print(true.shape)

                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  
0  December 31, 2017  
1  December 31, 2017  
2  December 30, 2017  
3  December 29, 2017  
4  December 25, 2017  
                                               title  \
0  As U.S. budget fight looms, Republicans flip t...   
1  U.S. military to accept t

In [17]:

display(fake.head())
display(true.head())

print(fake.shape)
print(true.shape)

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


(23481, 4)
(21417, 4)


In [18]:
print(fake.duplicated().sum())
print(true.duplicated().sum())

3
206


In [19]:
fake.drop_duplicates(inplace=True)
true.drop_duplicates(inplace=True)
print(fake.duplicated().sum())
print(true.duplicated().sum())

0
0


In [20]:
fake.isnull().sum() 
true.isnull().sum()

title      0
text       0
subject    0
date       0
dtype: int64

In [21]:
# Add label column to identify fake news as 0
fake["label"] = 0

# Add label column to identify real news as 1
true["label"] = 1

# Combine fake and true news datasets into one dataframe
data = pd.concat([fake, true], axis=0)

# Shuffle the dataset and reset the index
# to mix fake and real news randomly
data = data.sample(frac=1).reset_index(drop=True)

# Display first 5 rows of the final dataset
data.head()

,title,text,subject,date,label
0,GOVERNMENT SANTA CANDIDATE Gets Another Huge E...,"The USPS is drowning in debt, has lost every o...",left-news,"Nov 12, 2015",0
1,Obama to propose $200 million to battle Islami...,WASHINGTON (Reuters) - President Barack Obama ...,politicsNews,"February 9, 2016",1
2,TRUMP OBLITERATES “Phony Vietnam Con-Artist” D...,President Donald Trump continued his attack o...,politics,"Aug 8, 2017",0
3,"War crimes convict Praljak took cyanide, Dutch...",AMSTERDAM (Reuters) - A preliminary autopsy in...,worldnews,"December 1, 2017",1
4,The UNBELIEVABLE Reason Trump Pardoned Arpaio...,If Donald Trump hadn t proven himself to be th...,News,"August 28, 2017",0


In [22]:
data.duplicated().sum()

np.int64(0)

In [23]:
# Drop unnecessary columns
data = data.drop(["subject", "date"], axis=1)

# Display the updated dataset
data.head()

,title,text,label
0,GOVERNMENT SANTA CANDIDATE Gets Another Huge E...,"The USPS is drowning in debt, has lost every o...",0
1,Obama to propose $200 million to battle Islami...,WASHINGTON (Reuters) - President Barack Obama ...,1
2,TRUMP OBLITERATES “Phony Vietnam Con-Artist” D...,President Donald Trump continued his attack o...,0
3,"War crimes convict Praljak took cyanide, Dutch...",AMSTERDAM (Reuters) - A preliminary autopsy in...,1
4,The UNBELIEVABLE Reason Trump Pardoned Arpaio...,If Donald Trump hadn t proven himself to be th...,0


In [24]:
# Combine title and article text into one column
data["content"] = data["title"] + " " + data["text"]

# Display the new column
data[["content", "label"]].head()

,content,label
0,GOVERNMENT SANTA CANDIDATE Gets Another Huge E...,0
1,Obama to propose $200 million to battle Islami...,1
2,TRUMP OBLITERATES “Phony Vietnam Con-Artist” D...,0
3,"War crimes convict Praljak took cyanide, Dutch...",1
4,The UNBELIEVABLE Reason Trump Pardoned Arpaio...,0


In [25]:
data.isnull().sum()

title      0
text       0
label      0
content    0
dtype: int64

In [27]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


# Download NLP resources
nltk.download("stopwords")
nltk.download("wordnet")


# Define stopwords and lemmatizer
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


# Fill missing values
data["content"] = data["content"].fillna("")


# Convert text to string
data["content"] = data["content"].astype(str)


# Convert text to lowercase
data["content"] = data["content"].str.lower()


# Remove special characters and numbers
data["content"] = data["content"].apply(
    lambda x: re.sub(r'[^a-zA-Z\s]', '', x)
)


# Tokenization (split text into words)
data["content"] = data["content"].apply(lambda x: x.split())


# Remove stopwords
data["content"] = data["content"].apply(
    lambda words: [word for word in words if word not in stop_words]
)


# Lemmatization
data["content"] = data["content"].apply(
    lambda words: [lemmatizer.lemmatize(word) for word in words]
)


# Convert words list back to text
data["clean_content"] = data["content"].apply(
    lambda x: " ".join(x)
)


# Display result
data[["clean_content", "label"]].head()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\melgn\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\melgn\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,clean_content,label
0,government santa candidate get another huge en...,0
1,obama propose million battle islamist militant...,1
2,trump obliterates phony vietnam conartist dem ...,0
3,war crime convict praljak took cyanide dutch p...,1
4,unbelievable reason trump pardoned arpaio hurr...,0


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=5000)

# Convert text into vectors
X = vectorizer.fit_transform(data["clean_content"])

# Labels
y = data["label"]

# Check vector shape
print(X.shape)

(44689, 5000)


In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)